# Module 01 — LLM Basics

**Goal:** Make your first LLM API call from a Databricks notebook and understand every part of it.

This notebook runs on the **Databricks free tier** and talks to an LLM through the OpenAI-compatible API. The default provider is [GitHub Models](https://github.com/marketplace/models) — free with any GitHub account.

By the end you will know:
- What `messages` are and why roles matter
- What the raw API response object looks like
- Why we set `temperature=0` for deterministic tasks like SQL generation
- How to keep a multi-turn conversation

---
**Run each cell in order (Shift+Enter).**

## Step 1 — Install the `openai` client

The Databricks runtime does not ship with the `openai` package. Install it once per session, then restart Python so the import picks it up.

In [ ]:
%pip install --quiet openai
dbutils.library.restartPython()

## Step 2 — Set your parameters

We use **Databricks notebook widgets** so you never hard-code secrets. Run the cell below once — three input boxes appear at the top of the notebook.

| Widget | Default | What to do |
|--------|---------|------------|
| `API_KEY` | *(empty)* | Paste your GitHub token — create one at https://github.com/settings/tokens (default scopes are fine) |
| `BASE_URL` | `https://models.github.ai/inference` | Leave as-is for GitHub Models |
| `MODEL` | `openai/gpt-4o-mini` | Leave as-is, or pick another from https://github.com/marketplace?type=models |

> Want to use a different provider (OpenAI, Databricks Model Serving, Ollama, Groq)? Just change `BASE_URL`, `API_KEY`, and `MODEL` in the widgets — the rest of the notebook is identical.

In [ ]:
dbutils.widgets.text("API_KEY", "", "API key")
dbutils.widgets.text("BASE_URL", "https://models.github.ai/inference", "Base URL")
dbutils.widgets.text("MODEL", "openai/gpt-4o-mini", "Model")

API_KEY = dbutils.widgets.get("API_KEY")
BASE_URL = dbutils.widgets.get("BASE_URL")
MODEL = dbutils.widgets.get("MODEL")

if not API_KEY:
    raise ValueError("Paste your API key into the API_KEY widget at the top of the notebook.")

print(f"Model:    {MODEL}")
print(f"Endpoint: {BASE_URL}")
print("API key:  set")

## Step 3 — Build the client

The `openai` Python package works with **any** OpenAI-compatible endpoint. Point it at `BASE_URL` and you are done.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

## Step 4 — Your first API call

The minimum required fields are `model` and `messages`.

`messages` is a list of conversation turns. Each turn has:
- `role`: who is speaking — `"system"`, `"user"`, or `"assistant"`
- `content`: what they said

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is 2 + 2? Answer in one word."}
    ],
)

print(response.choices[0].message.content)

## Step 5 — Inspect the response

There's more in the response than just the text. The fields below show up again in later modules.

In [ ]:
print(f"model used:      {response.model}")
print(f"finish_reason:   {response.choices[0].finish_reason}")
print(f"content:         {response.choices[0].message.content}")
print(f"prompt tokens:   {response.usage.prompt_tokens}")
print(f"response tokens: {response.usage.completion_tokens}")

## Step 6 — The system prompt

The `system` role gives the model persistent instructions that apply to every user message. It's the most powerful knob for controlling model behavior.

In [ ]:
def ask(question: str, system: str = "") -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


print("No system prompt:")
print(ask("Tell me about Paris."))
print()
print("With system prompt (one sentence only):")
print(ask(
    "Tell me about Paris.",
    system="You are a tour guide. Answer every question in exactly one sentence.",
))

## Step 7 — Temperature: determinism vs creativity

`temperature=0` always picks the most likely next token → same question, same answer every time.  
`temperature=1` samples from the distribution → same question, different answers each run.

For SQL generation we always use `temperature=0`. For brainstorming or summarization you'd use 0.7–1.0.

> Free GitHub Models has rate limits. If you see a 429, wait a few seconds and retry.

In [ ]:
question = "Name one random country."

print("temperature=0 (deterministic) — should be the same every run:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
    )
    print(" ", r.choices[0].message.content)

print()
print("temperature=1 (creative) — should vary:")
for _ in range(3):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=1,
    )
    print(" ", r.choices[0].message.content)

## Step 8 — Multi-turn conversation

LLMs are **stateless** — they have no memory between API calls. You simulate memory by appending every turn to `messages` before the next call.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_message: str) -> str:
    messages.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    assistant_message = response.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_message})
    return assistant_message


print("Turn 1:", chat("My name is Alice."))
print()
print("Turn 2:", chat("What is my name?"))
print()
print(f"Messages in history: {len(messages)} (grows with every turn!)")

## Summary

| Concept | Key point |
|---------|----------|
| `messages` | The full conversation history sent on every call |
| `system` role | Persistent instructions that shape every response |
| `finish_reason` | Why the model stopped — important in the agentic loop (Module 04) |
| `temperature=0` | Deterministic output — use this for SQL generation |
| Stateless | LLMs don't remember — you must append history yourself |
| Provider-agnostic | Same Python code works with GitHub Models, OpenAI, Databricks Model Serving, Ollama, Groq — just change the widget values |

**Next:** Module 02 — how to get reliable *structured* output (JSON) from an LLM instead of free-form text.